# Challenges TCP

Donea Fernando-Emanuel
grupa 243

Ai nevoie de `scapy` și fișierele `.pcap` trimise pe Teams.

```
pip install scapy
```


---
## Challenge 2 — Reconstruct the Message

`challenge2.pcap` conține sute de pachete TCP de pe o rețea aglomerată.
Undeva în trafic, o singură conexiune a trimis un mesaj secret bucată cu bucată,
în mai multe pachete PSH — trimise **out of order**.

**Treaba ta:**
1. Găsește conexiunea care conține mesajul
2. Sortează pachetele ei în ordinea corectă
3. Concatenează payload-urile și citește mesajul

*Hint: sequence numbers există exact pentru a ști ordinea corectă.*


In [3]:
import scapy.config
scapy.config.conf.ipv6_enabled = False
from scapy.utils import rdpcap
from scapy.layers.inet import IP, TCP
from scapy.packet import Raw
from collections import defaultdict

packets = rdpcap("challenge2.pcap")
print(f"Total pachete: {len(packets)}")


Total pachete: 261


In [4]:
# Pasul 1: câte pachete PSH are fiecare conexiune?
# Conexiunea cu mesajul va ieși în evidență față de zgomot.

psh_per_conn = defaultdict(list)
for pkt in packets:
    if IP in pkt and TCP in pkt and Raw in pkt:
        if pkt[TCP].flags & 0x08:  # PSH flag
            conn = (pkt[IP].src, pkt[TCP].sport,
                    pkt[IP].dst, pkt[TCP].dport)
            psh_per_conn[conn].append(pkt)

print("Pachete PSH per conexiune:")
for conn, pkts in sorted(psh_per_conn.items(), key=lambda x: len(x[1])):
    print(f"  {conn[0]}:{conn[1]} → {conn[2]}:{conn[3]}  :  {len(pkts)} pachete PSH")


Pachete PSH per conexiune:
  10.0.1.42:54321 → 10.0.1.99:8080  :  6 pachete PSH
  143.53.112.160:45490 → 96.118.154.60:47612  :  7 pachete PSH
  37.100.228.19:24184 → 23.214.89.166:62270  :  7 pachete PSH
  116.172.41.240:39430 → 44.201.201.235:21078  :  7 pachete PSH
  139.106.100.207:25665 → 48.134.207.51:1950  :  7 pachete PSH
  31.157.204.51:50297 → 73.31.159.12:36836  :  7 pachete PSH
  67.122.151.34:20074 → 163.225.178.130:6557  :  7 pachete PSH
  174.49.124.183:45498 → 10.0.1.99:8080  :  8 pachete PSH
  127.235.208.26:64084 → 122.177.184.201:34786  :  10 pachete PSH
  26.167.69.232:50015 → 137.95.25.50:57164  :  10 pachete PSH
  76.116.237.248:48732 → 56.248.109.216:32425  :  10 pachete PSH
  149.239.215.13:30634 → 10.0.1.99:8080  :  10 pachete PSH
  157.197.238.244:9862 → 135.177.185.241:36793  :  11 pachete PSH
  16.7.131.122:22391 → 93.175.78.116:60921  :  11 pachete PSH
  78.172.194.163:44578 → 101.40.2.81:39235  :  13 pachete PSH
  27.141.103.9:3988 → 60.166.236.45:43506  :

In [24]:
# Pasul 2: alege conexiunea suspectă și uită-te la payload-urile ei

conn_aleasa = ("10.0.1.42", 54321, "10.0.1.99",8080)  # TODO: completează

flag_pkts = [pkt for pkt in psh_per_conn[conn_aleasa]]

print(f"Pachete găsite: {len(flag_pkts)}")
print()
print("Payload-uri în ordinea din pcap (greșită):")
for pkt in flag_pkts:
    print(f"  seq={pkt[TCP].seq}  payload={pkt[Raw].load}")


Pachete găsite: 6

Payload-uri în ordinea din pcap (greșită):
  seq=1482910345  payload=b'TCP_'
  seq=1482910349  payload=b'SEQU'
  seq=1482910357  payload=b'_MAS'
  seq=1482910353  payload=b'ENCE'
  seq=1482910365  payload=b'2026'
  seq=1482910361  payload=b'TER_'


In [25]:
# Pasul 3: sortează după sequence number și reconstruiește mesajul

# TODO: sortează flag_pkts după pkt[TCP].seq
flag_pkts.sort(key=lambda x:pkt[TCP].seq)

# TODO: concatenează payload-urile
mesaj = b"".join(pkt[Raw].load for pkt in flag_pkts)


print("Mesaj:", mesaj)


Mesaj: b'TCP_SEQU_MASENCE2026TER_'


---
## Challenge 3 — The Impersonator

Ai două fișiere:
- `legit.pcap` — o conexiune TCP legitimă
- `suspect.pcap` — o conexiune care arată similar, dar conține **4 anomalii**

Unele anomalii încalcă regulile protocolului TCP.
Altele sunt comportamente care nu ar apărea niciodată pe un sistem real.

**Treaba ta:** pentru fiecare verificare de mai jos, scrie condiția care detectează anomalia.
Nu îți spunem câte anomalii are fiecare celulă — poate una, poate niciuna.


In [7]:
legit   = rdpcap("legit.pcap")
suspect = rdpcap("suspect.pcap")

# Helper: afișează toate pachetele din ambele fișiere side-by-side
for label, pcap in [("LEGIT", legit), ("SUSPECT", suspect)]:
    print(f"=== {label} ===")
    for i, pkt in enumerate(pcap):
        if IP in pkt and TCP in pkt:
            flags = pkt[TCP].flags
            parts = []
            if flags & 0x02: parts.append("SYN")
            if flags & 0x10: parts.append("ACK")
            if flags & 0x01: parts.append("FIN")
            if flags & 0x08: parts.append("PSH")
            if flags & 0x04: parts.append("RST")
            payload = pkt[Raw].load if Raw in pkt else b""
            print(f"  [{i}] {'+'.join(parts):<12}  seq={pkt[TCP].seq:<12} "
                  f"ack={pkt[TCP].ack:<12}  payload={payload[:20]}")
    print()


=== LEGIT ===
  [0] SYN           seq=3847291033   ack=0             payload=b''
  [1] SYN+ACK       seq=2910482771   ack=3847291034    payload=b''
  [2] ACK           seq=3847291034   ack=2910482772    payload=b''
  [3] ACK+PSH       seq=3847291034   ack=2910482772    payload=b'SSzygm9D0U'
  [4] ACK+PSH       seq=3847291044   ack=2910482772    payload=b'gX0c4cTTVO'
  [5] ACK+PSH       seq=3847291054   ack=2910482772    payload=b'SZ6mSZ72sF'
  [6] ACK+FIN       seq=3847291064   ack=2910482772    payload=b''

=== SUSPECT ===
  [0] SYN           seq=0            ack=0             payload=b''
  [1] SYN+ACK       seq=1234567      ack=99            payload=b''
  [2] SYN+FIN       seq=1            ack=1234568       payload=b''
  [3] ACK+PSH       seq=1            ack=0             payload=b'GET / HTTP/1.0\r\n'
  [4] ACK+FIN       seq=16           ack=1234568       payload=b''



In [8]:
# Verificare 1 — ISN-ul clientului
# Regulă: sistemele moderne aleg ISN-ul random la fiecare conexiune nouă.
# Un ISN de exact 0 nu e neapărat interzis de protocol, dar nu apare niciodată în practică.

def get_syn(pcap):
    for pkt in pcap:
        if IP in pkt and TCP in pkt:
            if pkt[TCP].flags & 0x02 and not pkt[TCP].flags & 0x10:
                return pkt
    return None

syn_l = get_syn(legit)
syn_s = get_syn(suspect)

print(f"ISN legit:   {syn_l[TCP].seq}")
print(f"ISN suspect: {syn_s[TCP].seq}")
print()

# TODO: scrie condiția care detectează dacă ISN-ul e suspect
if syn_s[TCP].seq==0:
    print("ANOMALIE detectată: ISN-ul este egal cu zeor")


ISN legit:   3847291033
ISN suspect: 0

ANOMALIE detectată: ISN-ul este egal cu zeor


In [9]:
# Verificare 2 — SYN-ACK confirmă corect ISN-ul clientului?
# Regulă TCP: ack-ul din SYN-ACK trebuie să fie exact client_ISN + 1
# Dacă nu e, serverul fie nu a primit SYN-ul corect, fie e falsificat.

def get_synack(pcap):
    for pkt in pcap:
        if IP in pkt and TCP in pkt:
            if pkt[TCP].flags & 0x02 and pkt[TCP].flags & 0x10:
                return pkt
    return None

synack_l = get_synack(legit)
synack_s = get_synack(suspect)

print("Legit:")
print(f"  client ISN = {syn_l[TCP].seq}")
print(f"  SYN-ACK ack = {synack_l[TCP].ack}")
print()
print("Suspect:")
print(f"  client ISN = {syn_s[TCP].seq}")
print(f"  SYN-ACK ack = {synack_s[TCP].ack}")
print()

# TODO: verifică dacă SYN-ACK ack == client_ISN + 1 pentru fiecare
if synack_s[TCP].ack!= syn_s[TCP].seq+1:
     print("ANOMALIE detectată: syn-ack ul clientului nu confirma ISN-ul")


Legit:
  client ISN = 3847291033
  SYN-ACK ack = 3847291034

Suspect:
  client ISN = 0
  SYN-ACK ack = 99

ANOMALIE detectată: syn-ack ul clientului nu confirma ISN-ul


In [10]:
# Verificare 3 — Combinații de flag-uri imposibile
# Unele combinații de flag-uri nu pot apărea în TCP real.
# Un pachet nu poate fi în același timp SYN (deschide conexiunea)
# și FIN (închide conexiunea).

for label, pcap in [("LEGIT", legit), ("SUSPECT", suspect)]:
    print(f"=== {label} ===")
    for i, pkt in enumerate(pcap):
        if IP in pkt and TCP in pkt:
            flags = pkt[TCP].flags
            parts = []
            if flags & 0x02: parts.append("SYN")
            if flags & 0x10: parts.append("ACK")
            if flags & 0x01: parts.append("FIN")
            if flags & 0x08: parts.append("PSH")
            combo = "+".join(parts)

            # TODO: scrie condiția care detectează combinații imposibile
            imposibil=(flags & 0x02 and flags & 0x01) or (flags & 0x04 and flags & 0x10)

            print(f"  [{i}] {combo:<15} {'⚠ ANOMALIE' if imposibil else ''}")
    print()


=== LEGIT ===
  [0] SYN             
  [1] SYN+ACK         
  [2] ACK             
  [3] ACK+PSH         
  [4] ACK+PSH         
  [5] ACK+PSH         
  [6] ACK+FIN         

=== SUSPECT ===
  [0] SYN             
  [1] SYN+ACK         
  [2] SYN+FIN         ⚠ ANOMALIE
  [3] ACK+PSH         
  [4] ACK+FIN         



In [11]:
# Verificare 4 — ACK-ul din pachetele de date
# Regulă: după handshake, fiecare pachet al clientului trebuie să confirme
# că a primit SYN-ACK-ul serverului: ack = server_ISN + 1
# Dacă ack e greșit, fie clientul nu a procesat SYN-ACK, fie e o conexiune falsificată.

def get_server_isn(pcap):
    sa = get_synack(pcap)
    return sa[TCP].seq if sa else None

server_isn_l = get_server_isn(legit)
server_isn_s = get_server_isn(suspect)

for label, pcap, s_isn in [("LEGIT", legit, server_isn_l),
                             ("SUSPECT", suspect, server_isn_s)]:
    print(f"=== {label} (server ISN = {s_isn}) ===")
    for pkt in pcap:
        if IP in pkt and TCP in pkt and Raw in pkt:
            ack = pkt[TCP].ack
            asteptat = s_isn + 1

            # TODO: verifică dacă ack == server_ISN + 1
            if ack != asteptat:
                print(f"  ANOMALIE: ack={ack}, așteptat={asteptat}")
            else:
                print(f"  OK: ack={ack}")

            print(f"  ack={ack}  așteptat={asteptat}  ???")
    print()


=== LEGIT (server ISN = 2910482771) ===
  OK: ack=2910482772
  ack=2910482772  așteptat=2910482772  ???
  OK: ack=2910482772
  ack=2910482772  așteptat=2910482772  ???
  OK: ack=2910482772
  ack=2910482772  așteptat=2910482772  ???

=== SUSPECT (server ISN = 1234567) ===
  ANOMALIE: ack=0, așteptat=1234568
  ack=0  așteptat=1234568  ???



---
## Challenge 4 — Follow the Stream

`challenge4.pcap` conține 5 conexiuni TCP simultane, toate amestecate.
Patru transportă zgomot. Una transportă un mesaj secret.

Wireshark are o funcție numită "Follow TCP Stream" care face exact ce trebuie să faci tu acum:
grupează pachetele după conexiune, le pune în ordine, și reconstituie conversația.

**Treaba ta:** implementează Follow TCP Stream de mână.


In [26]:
packets4 = rdpcap("challenge4.pcap")
print(f"Total pachete: {len(packets4)}")
print()

# Pasul 1: identifică toate conexiunile unice
# Normalizăm direcția — același stream apare în ambele sensuri
conexiuni4 = set()
for pkt in packets4:
    if IP in pkt and TCP in pkt:
        conn = tuple(sorted([(pkt[IP].src, pkt[TCP].sport),
                              (pkt[IP].dst, pkt[TCP].dport)]))
        conexiuni4.add(conn)

print(f"Conexiuni găsite: {len(conexiuni4)}")
for c in sorted(conexiuni4):
    print(f"  {c[0][0]}:{c[0][1]} ↔ {c[1][0]}:{c[1][1]}")


Total pachete: 52

Conexiuni găsite: 5
  102.106.224.208:7438 ↔ 155.129.199.115:51736
  154.16.190.254:4020 ↔ 188.167.77.59:16732
  173.56.74.158:31984 ↔ 65.221.42.235:12729
  192.168.5.20:7777 ↔ 192.168.5.7:61234
  33.157.58.119:60761 ↔ 65.50.81.115:41704


In [28]:
# Pasul 2: pentru fiecare conexiune, reconstruiește stream-ul
# Adună pachetele PSH care aparțin conexiunii, sortează după seq, concatenează payload.

for conn in sorted(conexiuni4):
    (ip1, p1), (ip2, p2) = conn

    # TODO: filtrează pachetele PSH care aparțin acestei conexiuni
    # Un pachet aparține conexiunii dacă:
    #   (src==ip1 AND sport==p1) SAU (src==ip2 AND sport==p2)
    psh_pkts = []
    for pkt in packets4:

        if (pkt[IP].src == ip1 and pkt[TCP].sport == p1) or (pkt[IP].src == ip2 and pkt[TCP].sport == p2):
            psh_pkts.append(pkt)

    # TODO: sortează după sequence number
    psh_pkts.sort(key=lambda x: x[TCP].seq)

    # TODO: concatenează payload-urile
    stream_data = b"".join(pkt[Raw].load for pkt in psh_pkts if Raw in pkt)

    print(f"{ip1}:{p1} ↔ {ip2}:{p2}  →  {stream_data}")



102.106.224.208:7438 ↔ 155.129.199.115:51736  →  b'r98kaQ0x3ZHeezlu4qxRhOpn0reIHwGzuG1Sg6DPNPz4gfSvw67TiT0XFw91lVRkY'
154.16.190.254:4020 ↔ 188.167.77.59:16732  →  b'Ap4dZW8ypUhEqNSxep1VkM41qidZl36xFhKd8zrqySIcWLsp771ycvpXIolPWu1phQ'
173.56.74.158:31984 ↔ 65.221.42.235:12729  →  b'QqyqDh6WmK6K5W1M7yhYBISrywgTx'
192.168.5.20:7777 ↔ 192.168.5.7:61234  →  b'FOLLOW_THE_STREAM_2026'
33.157.58.119:60761 ↔ 65.50.81.115:41704  →  b'RaBlsbkl2CGdOcuV2xL79OySkOBAzFPw4HCgFOLHpGyvJDtcIDSbJLflv8Exz2UOQuptXO'


In [22]:
# Pasul 3: care stream conține flag-ul?
# Odată ce ai toate stream-urile reconstruite, flag-ul e evident.
# Scrie-l mai jos:

flag='FOLLOW_THE_STREAM_2026'
print(flag)



FOLLOW_THE_STREAM_2026
